# 01 Embedding

本章目标：

1. 理解 Embedding 的作用
2. 理解文本如何映射为向量
3. 掌握 Cosine Similarity
4. 理解 Dense Embedding 与 Sparse Representation 的区别
5. 使用真实 Embedding 模型完成文本向量化
6. 手写一个最小语义检索器
7. 理解 Query Embedding 与 Document Embedding
8. 理解为什么 Embedding 模型不能随意切换

In [33]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

import sentence_transformers

print("sentence-transformers version:", sentence_transformers.__version__)

Python executable:
c:\Users\Administrator\miniconda3\envs\agent_project\python.exe

Python version:
3.11.17 | packaged by Anaconda, Inc. | (main, Oct  1 2026, 20:16:54) [MSC v.1942 64 bit (AMD64)]
sentence-transformers version: 6.2.0


## 1. 加载第一个 Embedding 模型

这里先使用 `BAAI/bge-small-zh-v1.5`。

选择它的原因：

- 支持中文
- 模型体积较小
- 适合入门实验
- 可以完成语义检索任务
- 后面便于和更大的 Embedding 模型进行对比

In [34]:
from sentence_transformers import SentenceTransformer

model_name = "BAAI/bge-small-zh-v1.5"
model = SentenceTransformer(model_name)

print("model loaded successfully")
print(model)

print("embeding dimension: ")
print(model.get_embedding_dimension())

Loading weights: 100%|██████████| 71/71 [00:00<00:00, 466.85it/s]


model loaded successfully
SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 512, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)
embeding dimension: 
512


# 2.第一次生成文本 embedding

In [35]:
sentences = [
    "持续强降雨容易诱发滑坡",
    "长时间降水会增加边坡失稳风险",
    "Python是一种编程语言",
    "光伏功率受到太阳辐射影响"
]

embeddings = model.encode(sentences)
print(type(embeddings))
print(embeddings.shape)


print(embeddings[0][:10])

print("第一句话的向量维度：", len(embeddings[0]))

<class 'numpy.ndarray'>
(4, 512)
[-0.0666433  -0.01446033  0.01246678  0.06323322 -0.04639034 -0.07236915
 -0.00772836  0.04791742  0.04613344  0.03981128]
第一句话的向量维度： 512



# 3. cosine similarity(余弦相似度)
Embedding 将文本转换为向量后，需要定义“两个向量有多相似”。

本节手写余弦相似度，并比较不同文本之间的语义相似程度。

In [36]:
import numpy as np

def cosine_similarity(v1, v2):

    # norm_v1 = np.linalg.norm(v1)
    return np.dot(v1, v2) / (np.sqrt(np.sum(v1 ** 2)) * np.sqrt(np.sum(v2 ** 2)))

for i in range(len(embeddings)):
    score = cosine_similarity(embeddings[0], embeddings[i])
    print(f"{sentences[0]} <-> {sentences[i]}" f" | similarity: {score:.3f}")

持续强降雨容易诱发滑坡 <-> 持续强降雨容易诱发滑坡 | similarity: 1.000
持续强降雨容易诱发滑坡 <-> 长时间降水会增加边坡失稳风险 | similarity: 0.768
持续强降雨容易诱发滑坡 <-> Python是一种编程语言 | similarity: 0.224
持续强降雨容易诱发滑坡 <-> 光伏功率受到太阳辐射影响 | similarity: 0.453


# 4. 相似度矩阵
计算所有文本两两之间的余弦相似度，观察语义相近文本是否聚集在一起。

In [37]:
from sentence_transformers import SimilarityFunction


n = len(sentences)

Similarity_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        Similarity_matrix[i, j] = cosine_similarity(embeddings[i], embeddings[j])
Similarity_matrix

array([[1.        , 0.76827347, 0.22444899, 0.45263121],
       [0.76827347, 1.        , 0.21072961, 0.44406176],
       [0.22444899, 0.21072961, 1.        , 0.2464491 ],
       [0.45263121, 0.44406176, 0.2464491 , 1.00000024]])

In [38]:
import pandas as pd

df_similarity = pd.DataFrame(
    Similarity_matrix,
    index=sentences,
    columns=sentences
)

df_similarity

,持续强降雨容易诱发滑坡,长时间降水会增加边坡失稳风险,Python是一种编程语言,光伏功率受到太阳辐射影响
持续强降雨容易诱发滑坡,1.000000,0.768273,0.224449,0.452631
长时间降水会增加边坡失稳风险,0.768273,1.000000,0.210730,0.444062
Python是一种编程语言,0.224449,0.210730,1.000000,0.246449
光伏功率受到太阳辐射影响,0.452631,0.444062,0.246449,1.000000


# 5. 最小语义检索器
目标：

1. 构建一个小型知识库
2. 将文档提前转换为 Embedding
3. 将用户 Query 转成 Embedding
4. 计算 Query 与所有文档的相似度
5. 返回 Top-K 最相关文档

In [39]:
documents = [
    "持续降雨会导致土体含水率增加，并可能提高孔隙水压力。",
    "坡度是影响滑坡稳定性的重要地形因素。",
    "Python广泛用于数据分析和机器学习。",
    "太阳辐射是影响光伏发电功率的重要气象因素。",
    "高温天气可能降低光伏组件的转换效率。",
    "地下水位升高可能降低边坡稳定性。",
    "植被覆盖能够在一定程度上改善坡体表层稳定性。",
    "短时强降水容易诱发山洪和地质灾害。",
    "PyTorch是常用的深度学习框架。",
    "云量变化会直接影响到达地表的太阳辐射。"
]
document_embeddings = model.encode(documents,normalize_embeddings=True)
print(document_embeddings.shape)
# 查询
query = "降雨为什么会提高滑坡发生风险？"
query_embedding = model.encode(query)
# 计算余弦相似度
scores = []
for i,document_embedding in enumerate(document_embeddings):
    score = cosine_similarity(query_embedding, document_embedding)
    
    scores.append({"index": i,  "text": documents[i],"score": float(score)})
# 排序
scores.sort(key=lambda x: x["score"], reverse=True)

scores[:3]

top_k = 5
for rank,result in enumerate(scores[:top_k],start=1):
    print(f"Top_k {rank}")
    print(f"Score: {result['score']:.4f}")
    print(f"Text: {result['text']}")
    print("-"*60)

(10, 512)
Top_k 1
Score: 0.7072
Text: 地下水位升高可能降低边坡稳定性。
------------------------------------------------------------
Top_k 2
Score: 0.6948
Text: 坡度是影响滑坡稳定性的重要地形因素。
------------------------------------------------------------
Top_k 3
Score: 0.6883
Text: 持续降雨会导致土体含水率增加，并可能提高孔隙水压力。
------------------------------------------------------------
Top_k 4
Score: 0.6579
Text: 短时强降水容易诱发山洪和地质灾害。
------------------------------------------------------------
Top_k 5
Score: 0.5566
Text: 植被覆盖能够在一定程度上改善坡体表层稳定性。
------------------------------------------------------------


# 6.封装检索器


In [40]:
def semantic_search(query,documents, document_embeddings,model,top_k):
    query_embedding = model.encode(query, normalize_embeddings=True)

    scores = []
    for i,document_embedding in enumerate(document_embeddings):
        score = np.dot(query_embedding, document_embedding)

        scores.append({"index": i,  "text": documents[i],"score": float(score)})
    scores.sort(key=lambda x: x["score"], reverse=True)
    return scores[:top_k]

# 测试
# query = "地下水变化会不会影响滑坡？"

# results = semantic_search(
#     query=query,
#     documents=documents,
#     document_embeddings=document_embeddings,
#     model=model,
#     top_k=3
# )

# for result in results:
#     print(
#         f"{result['score']:.4f} | "
#         f"{result['text']}"
#     )
query_1 = "持续降雨会不会引发滑坡？"

query_2 = "连续的降水过程是否会提高坡体失稳概率？"
results_1 = semantic_search(
    query_1,
    documents,
    document_embeddings,
    model,
    top_k=3
)

results_2 = semantic_search(
    query_2,
    documents,
    document_embeddings,
    model,
    top_k=3
)
print("Query 1:", query_1)

for r in results_1:
    print(f"{r['score']:.4f} | {r['text']}")

print("\n" + "=" * 80 + "\n")

print("Query 2:", query_2)

for r in results_2:
    print(f"{r['score']:.4f} | {r['text']}")

Query 1: 持续降雨会不会引发滑坡？
0.6847 | 持续降雨会导致土体含水率增加，并可能提高孔隙水压力。
0.6767 | 地下水位升高可能降低边坡稳定性。
0.6509 | 坡度是影响滑坡稳定性的重要地形因素。


Query 2: 连续的降水过程是否会提高坡体失稳概率？
0.7557 | 地下水位升高可能降低边坡稳定性。
0.6892 | 持续降雨会导致土体含水率增加，并可能提高孔隙水压力。
0.6371 | 坡度是影响滑坡稳定性的重要地形因素。


# 6.Normalization、Cosine Similarity 与 Dot Product

本节目标：

1. 理解向量归一化（Normalization）
2. 理解 Cosine Similarity 与 Dot Product 的关系
3. 验证归一化后 Dot Product = Cosine Similarity
4. 理解为什么 Embedding 检索经常使用 normalize_embeddings=True

In [41]:
def normalize_vector(vector):
    vector = np.array(vector,dtype=float)
    norm = np.linalg.norm(vector)
    return vector/norm

a = np.array([3, 4])

a_normalized = normalize_vector(a)

print("原向量:", a)
print("原向量长度:", np.linalg.norm(a))

print()

print("归一化向量:", a_normalized)
print("归一化后长度:", np.linalg.norm(a_normalized))


a = np.array([1, 2, 3], dtype=float)
b = np.array([2, 1, 4], dtype=float)

cos_score = cosine_similarity(a, b)

a_norm = normalize_vector(a)
b_norm = normalize_vector(b)

dot_score = np.dot(a_norm, b_norm)

print("Cosine Similarity :", cos_score)
print("Normalized Dot    :", dot_score)

原向量: [3 4]
原向量长度: 5.0

归一化向量: [0.6 0.8]
归一化后长度: 1.0
Cosine Similarity : 0.933138949631687
Normalized Dot    : 0.933138949631687


## 7. Embedding 模型选择

模型选择需要综合考虑：

- 语言支持
- 检索效果
- Embedding维度
- 模型参数量
- 推理速度
- 显存/内存消耗
- 最大输入长度
- 是否支持 Dense / Sparse / Multi-vector
- 部署成本

In [42]:
print("Model:", model_name)
print("Embedding dimension:", model.get_embedding_dimension())
print("Max sequence length:", model.max_seq_length)

Model: BAAI/bge-small-zh-v1.5
Embedding dimension: 512
Max sequence length: 512


### 当前模型记录

- Model: BAAI/bge-small-zh-v1.5
- Embedding dimension: 512
- Max sequence length: 512
- Language: Chinese
- Retrieval type: Dense Retrieval